In [1]:
# Why: Add the project root to Python's import path so notebook cells can use project modules.
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [2]:
# Why: Import the tools needed to read PDFs, clean and split text, create metadata, and prepare later RAG steps.
# Standard library
import os

# Environment
from dotenv import load_dotenv

# PDF processing
import fitz

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LLM
from openai import OpenAI

from ingestion.preprocess import clean_text
from ingestion.chunking import split_into_chunks
from ingestion.metadata import create_metadata

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Why: Confirm which Python environment runs the notebook before installing or importing packages.
import sys

print(sys.executable)

/home/codespace/.python/current/bin/python


In [4]:
# Why: Verify that pip is available in this notebook's active Python environment.
import subprocess
subprocess.run([sys.executable, "-m", "pip", "--version"])

pip 26.0.1 from /usr/local/python/3.12.1/lib/python3.12/site-packages/pip (python 3.12)


CompletedProcess(args=['/home/codespace/.python/current/bin/python', '-m', 'pip', '--version'], returncode=0)

In [5]:
# Why: Load credentials and configure the API client for any later language-model calls.
load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [6]:
import os

print(os.getcwd())

/workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/Notebooks


In [7]:
from pathlib import Path

pdf_folder = (Path.cwd() / "../storage/raw").resolve()

print(pdf_folder)

/workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/storage/raw


In [8]:

pdf_folder = Path("../storage/raw")

print("Exists:", pdf_folder.exists())
print("Absolute:", pdf_folder.resolve())

print("\nPDF files:")

for pdf in pdf_folder.glob("*.pdf"):
    print(pdf.name)

Exists: True
Absolute: /workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/storage/raw

PDF files:
Cosmic Origins Research Plan.pdf
In modern scientific terms.pdf


In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "storage" / "raw"

print(RAW_DIR)

/workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/storage/raw


In [10]:
for pdf_path in RAW_DIR.glob("*.pdf"):
    ...

In [11]:

pdf_folder = Path("../storage/raw")

for pdf_path in pdf_folder.glob("*.pdf"):
    print(f"Processing: {pdf_path.name}")

    doc = fitz.open(pdf_path)

    print("Pages:", len(doc))

Processing: Cosmic Origins Research Plan.pdf


Pages: 10
Processing: In modern scientific terms.pdf
Pages: 9


In [12]:
import json

project_root = Path.cwd().parent

raw_dir = project_root / "storage" / "raw"
processed_dir = project_root / "storage" / "processed"

processed_dir.mkdir(parents=True, exist_ok=True)

output_file = processed_dir / "chunks.json"

# -----------------------------
# Process all PDFs
# -----------------------------

all_chunks = []

pdf_files = sorted(raw_dir.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF(s)\n")

for pdf_path in pdf_files:

    print(f"Processing: {pdf_path.name}")

    # -------------------------
    # Extract text
    # -------------------------

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    doc.close()

    # -------------------------
    # Clean
    # -------------------------

    cleaned_text = clean_text(text)

    # -------------------------
    # Chunk
    # -------------------------

    chunks = split_into_chunks(cleaned_text)

    print(f"   Created {len(chunks)} chunks")

    # -------------------------
    # Metadata
    # -------------------------

    for chunk_index, chunk in enumerate(chunks):

        all_chunks.append(
            {
                "id": f"{pdf_path.stem}_{chunk_index}",
                "source": pdf_path.name,
                "chunk_id": chunk_index,
                "text": chunk,
            }
        )

print(f"\nTotal chunks: {len(all_chunks)}")

# -----------------------------
# Save JSON
# -----------------------------

with output_file.open("w", encoding="utf-8") as f:
    json.dump(
        all_chunks,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f"\nSaved to:\n{output_file}")

Found 2 PDF(s)

Processing: Cosmic Origins Research Plan.pdf
   Created 53 chunks
Processing: In modern scientific terms.pdf
   Created 55 chunks

Total chunks: 108

Saved to:
/workspaces/LLM---ZOOMCAMP---RAG---PROJECT--1-/storage/processed/chunks.json


In [13]:
print(f"\n{pdf_path.name}")

print("Original characters:", len(text))

print("Cleaned characters:", len(cleaned_text))

print("Chunks:", len(chunks))


In modern scientific terms.pdf
Original characters: 33019
Cleaned characters: 32335
Chunks: 55


In [14]:
print(f"Total chunks: {len(all_chunks)}")
print(f"Unique PDFs: {len({c['source'] for c in all_chunks})}")
print(f"First chunk preview:\n{all_chunks[0]['text'][:300]}")

Total chunks: 108
Unique PDFs: 2
First chunk preview:
Comprehensive Cosmological Synthesis: Scientific, Philosophical, and Religious Perspectives on Cosmic Origins The inquiry into the origin of the universe represents one of the most profound intersections of empirical science, formal philosophy, and cultural narrative. Across millennia, human civiliz
